# MODE 2 — M2_F03 SCENOGRAPHY DOCK — PRODUCTION

> Phase 8 — Dual Pipeline Doctrine — v1.0.0

```
INPUT  : IN_GLB_DECOR/decor.glb + IN_GLB_AVATAR/avatar_validated.glb
OUTPUT : OUT_SCENE/scene_m2.blend
```

**4 opérations Blender** : Décor | Avatar | Shadow Catcher Y=0 | HDRi

In [ ]:
# ── CELLULE 0 — CONFIGURATION OPÉRATEUR ──────────────────────
DECOR_FILE   = None   # None = auto IN_GLB_DECOR/*.glb
AVATAR_FILE  = None   # None = auto IN_GLB_AVATAR/*.glb
HDRI_FILE    = None   # None = auto-détection repo | chemin explicite
SKIP_HDRI    = False  # True = ciel neutre par défaut
SHADOW_SIZE  = 50.0   # Taille shadow catcher en mètres
BLENDER_PATH = None   # None = auto-détection PATH
DRY_RUN      = False
VERBOSE      = True

In [ ]:
# ── CELLULE 1 — LANCEMENT ─────────────────────────────────────
import subprocess, sys
from pathlib import Path

script = Path("EXO_M2_F03_SCENOGRAPHY.py")
cmd = [sys.executable, str(script)]

if DECOR_FILE:   cmd += ["--decor",  DECOR_FILE]
if AVATAR_FILE:  cmd += ["--avatar", AVATAR_FILE]
if HDRI_FILE:    cmd += ["--hdri",   HDRI_FILE]
if SKIP_HDRI:    cmd.append("--skip-hdri")
if BLENDER_PATH: cmd += ["--blender", BLENDER_PATH]
cmd += ["--shadow-size", str(SHADOW_SIZE)]
if DRY_RUN:  cmd.append("--dry-run")
if VERBOSE:  cmd.append("--verbose")

print(f"Commande : {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False, text=True)
print(f"\nCode retour : {result.returncode}")

In [ ]:
# ── CELLULE 2 — RAPPORT ───────────────────────────────────────
import json
from pathlib import Path

report_path = Path("../OUT_REPORT/m2_f03_report.json")
if not report_path.exists():
    print("Rapport introuvable")
else:
    with open(report_path) as f:
        r = json.load(f)
    icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"{icon} STATUS : {r['status']}")
    br = r.get("blender_run", {}).get("internal", {})
    if br:
        print(f"   Décor importé   : {'✅' if br.get('decor_imported') else '⚠️ '} {br.get('decor_imported')}")
        print(f"   Avatar importé  : {'✅' if br.get('avatar_imported') else '⚠️ '} {br.get('avatar_imported')}")
        print(f"   Shadow catcher  : ✅ {br.get('shadow_catcher')}")
        print(f"   HDRi            : {'✅' if br.get('hdri_applied') else '⚠️ '} {br.get('hdri_applied')}")
        print(f"   Objets scène    : {br.get('object_count')}")
    out = r.get("outputs", {})
    if out.get("blend"):
        blend_path = Path(out["blend"])
        size_mb = blend_path.stat().st_size / (1024*1024) if blend_path.exists() else 0
        print(f"   Output .blend   : {blend_path.name} ({size_mb:.2f} MB)")
    if r["status"] == "SUCCESS":
        print("\n   ──────────────────────────────────")
        print("   ✅ Prêt pour M2_F04 ─► OUT_SCENE/")
        print("   Transférer OUT_SCENE/ vers 10_M2_F04_PHOTOGRAPHY/IN_*/")